# Analisi dati per effetto Hall

A grandi linee:
1. caratterizzare l'**uniformità del campo magnetico** nel traferro
2. caratterizzare l'andamento del **campo magnetico** nel traferro al **variare della corrente** in ingresso
3. caratterizzare **tensione di Hall al variare della corrente** $V_H(i)$ (ferromagnete spento + 5x2 valori del campo magnetico)
4. caratterizzare **tensione di Hall al variare del campo magnetico** $V_H(B)$ (ferromagnete spento + 3x2 valori di corrente)
5. valutare **mobilità** dei portatori di carica nel materiale

+ cose facolative (?)

### utils

In [103]:
import numpy as np
from utils import meanCalc, fitPlotter, testZ, MeanError, Zscore, Amprobe, Keithley, Teslameter, texTabler
import ROOT

## Uniformità campo magnetico

Facendo misure del campo magnetico con una sonda di Hall vogliamo studiare l'uniformità di $B$ all'interno del traferro del ferromagnete.

In generale, la semidifferenza tra il valore massimo e il valore minimo del campo magnetico (all'interno della regione di omogeneità) sarà il limite minimo dell'errore su tutte le misure di campo magnetico.

In [104]:
# arrays of distances 
posx = np.arange(5)*10
posy = np.arange(5)*10

# matrix of B values in mT
Bu = np.array([[176.20,204.5,202.0,203.6,164.16],
               [205.98,234.51,233.37,234.15,211.61],
               [215.00,236.,236.47,236.65,209.43],
               [204.67,235.5,235.5,235.3,207.4],
               [176.24,200.,201.10,202.05,175.02]])

Can2d_u = ROOT.TCanvas("c2d", "Uniformità campo magnetico")
H2d_u = ROOT.TH2F("h2d_u", "Uniformità campo magnetico; x [mm]; y [mm]; B [T]" ,5,0,5, 5,0,5)

for i in range(5):
    for j in range(5):
        H2d_u.SetBinContent(i+1,j+1,float(Bu[i][j]))

H2d_u.Draw("LEGO2")
Can2d_u.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2d
Warning in <TROOT::Append>: Replacing existing TH1: h2d_u (Potential memory leak).


Dopo lo studio di uniformità del campo magnetico, si stabilisce in quale zona effettuare le misure di campo magnetico. In tale zona, si prende la semidispersione di B come stima della risoluzione del campo magnetico.

In [105]:
# we calculate the semidifference to get an estimate on the error on the measurement of B

B_max = 236.65 # mT
B_min = 233.4  # mT
resB = (B_max - B_min)/2 # mT
errresB = np.sqrt(Teslameter([B_max])**2 + Teslameter([B_min])**2) / 2
print(resB, "in mT", errresB) 

1.625 in mT [9.01667838]


In [106]:
# i think that this will be the most complicated use of texTabler
for i in range(0,5):
    texTabler([Bu[i], np.ones(5)*i*10, np.arange(5)*10],[Teslameter(Bu[i]), np.ones(5), np.ones(5)], ["B", "x", "y"])

$B$ & $\delta B$ & $x$ & $\delta x$ & $y$ & $\delta y$ \\
\hline
176 & 10 & 0.0 & 1.0 & 0.0 & 1.0\\
205 & 11 & 0.0 & 1.0 & 10.0 & 1.0\\
202 & 11 & 0.0 & 1.0 & 20.0 & 1.0\\
204 & 11 & 0.0 & 1.0 & 30.0 & 1.0\\
164 & 9 & 0.0 & 1.0 & 40.0 & 1.0\\
$B$ & $\delta B$ & $x$ & $\delta x$ & $y$ & $\delta y$ \\
\hline
206 & 11 & 10.0 & 1.0 & 0.0 & 1.0\\
235 & 13 & 10.0 & 1.0 & 10.0 & 1.0\\
233 & 13 & 10.0 & 1.0 & 20.0 & 1.0\\
234 & 13 & 10.0 & 1.0 & 30.0 & 1.0\\
212 & 12 & 10.0 & 1.0 & 40.0 & 1.0\\
$B$ & $\delta B$ & $x$ & $\delta x$ & $y$ & $\delta y$ \\
\hline
215 & 12 & 20.0 & 1.0 & 0.0 & 1.0\\
236 & 13 & 20.0 & 1.0 & 10.0 & 1.0\\
236 & 13 & 20.0 & 1.0 & 20.0 & 1.0\\
237 & 13 & 20.0 & 1.0 & 30.0 & 1.0\\
209 & 11 & 20.0 & 1.0 & 40.0 & 1.0\\
$B$ & $\delta B$ & $x$ & $\delta x$ & $y$ & $\delta y$ \\
\hline
205 & 11 & 30.0 & 1.0 & 0.0 & 1.0\\
236 & 13 & 30.0 & 1.0 & 10.0 & 1.0\\
236 & 13 & 30.0 & 1.0 & 20.0 & 1.0\\
235 & 13 & 30.0 & 1.0 & 30.0 & 1.0\\
207 & 11 & 30.0 & 1.0 & 40.0 & 1.0\\
$B$ & $\de

## Campo magnetico prodotto al variare della corrente

Tralasciando la prima curva di salita (da $0$ a $i_\text{max}$) misuriamo due salite e due discese complete (da $i_\text{max}$ a $-i_\text{max}$, e viceversa) prendendo 10 dati per ogni curva. Sostanzialmente, vogliamo verificare la regione di linearità in cui (dopo) vogliamo svolgere il resto dell'esperienza.

Escluse le zone di saturazione vogliamo un fare un fit lineare su ogni curva: per ognuna delle due coppie (due curve di salita, e due curve di discesa) stimiamo coi valori medi di $m$ e $q$ il vero coefficiente angolare e la vera quota (*l'errore sulla quota lo stimiamo con la semidifferenza tra i due valori*).
Solo dopo mediamo i risultati ottenuti per le curve di salita e quelle di discesa (*errore sulla quota sempre dato dalla semidifferenza*).

In [107]:
# we would like to see the hystheresis loop
# answer: i believe that the greatest contribution is given by resB (see cell above), however we need to check with the teslamete

# UNITS: B in mT and i in A

# first negative run

B1 = np.array([355.2, 322.5, 279.3, 243, 200.4, 159, 104.5, 59.8, 9.3, -47.7, -87.7, -136.4, -183.3, -228.5, -269.9, -314.1, -349.4, -383.8]) #mT
i1 = np.array([1.602, 1.406, 1.18, 0.999, 0.8, 0.619, 0.393, 0.208, 0, -0.234, -0.4, -0.603, -0.802, -1.011, -1.201, -1.419, -1.604, -1.805]) #A
errB1 = Teslameter(B1)
erri1 = Amprobe(i1, unit = "A")

# first positive run
B2 = np.array([-358.1, -322.4, -282.9, -244.4, -201.6, -154.4, -94.7, -58.9, -8.7, 40.2, 89.6, 138.9, 189.1, 231.9, 268.8, 310.6, 350.7, 381.7]) #mT
i2 = np.array([-1.605, -1.4, -1.195, -1.007, -0.806, -0.599, -0.353, -0.206, 0, 0.202, 0.405, 0.611, 0.825, 1.025, 1.198, 1.403, 1.615, 1.798]) #A
errB2 = Teslameter(B2)
erri2 = Amprobe(i2,unit = "A")

# second negative run
B3 = np.array([356.3, 319.6, 281.7, 243.8, 198, 157.7, 108.3, 56.7, 8.9, -44.5, -88.4, -141.8, -185.5, -227.3, -272.3, -313.6, -349.9, -383.8]) #mT
i3 = np.array([1.599, 1.383, 1.192, 1.003, 0.788, 0.611, 0.409, 0.197, 0, -0.219, -0.4, -0.621, -0.808, -1.001, -1.213, -1.416, -1.606, -1.807]) #A
errB3 = Teslameter(B3)
erri3 = Amprobe(i3, unit = "A") 

# second positive run 
# we have five more data as we wanted to see the magnetic saturation
B4 = np.array([-358.7, -325.1, -284.8, -240.8, -202.5, -156, -106.2, -56.4, -8.7, 42.5, 87.2, 137.5, 190.2, 227.7, 272.7, 321.9, 349.5, 381.7, 414.9, 452, 476.7, 495.4, 499.9]) #mT
i4 = np.array([-1.61, -1.409, -1.201, -0.987, -0.808, -0.603, -0.399, -0.195, 0, 0.21, 0.395, 0.602, 0.83, 1, 1.208, 1.456, 1.602, 1.798, 2.013, 2.318, 2.627, 2.9, 2.977]) #A
errB4 = Teslameter(B4)
erri4 = Amprobe(i4, unit = "A")

# here we fix the minimum uncertainty on the magnetic fields using the semidifference
errB1[errB1 < resB] = resB
errB2[errB2 < resB] = resB
errB3[errB3 < resB] = resB
errB4[errB4 < resB] = resB

# we have discarded some data (it was outside of the linearity regime)
magCurrPlotter = fitPlotter("MagneticFieldVsCurrent")
param1 = magCurrPlotter.addGraph(i1, B1, erri1, errB1, title="Run Down 1; I [A]; B[mT]", xrange=[-1.250,+1.250])
param2 = magCurrPlotter.addGraph(i2, B2, erri2, errB2, title="Run Up 1; I [A]; B[mT]", xrange=[-1.250,+1.250])
param3 = magCurrPlotter.addGraph(i3, B3, erri3, errB3, title="Run Down 2; I [A]; B[mT]", xrange=[-1.250,+1.250])
param4 = magCurrPlotter.addGraph(i4, B4, erri4, errB4, title="Run Up 2; I [A]; B[mT]", xrange=[-1.250,+1.250])
magCurrPlotter.drawCanvas(legend=False,dimX=1500,dimY=750)
magCurrPlotter.saveCanvas("outputs/MagneticFieldVsCurrent.png")


--- fit Results for: Run Down 1; I [A]; B[mT] ---
Function: pol1
Chi2/NDF: 1.2194 / 11
p-value:  0.9999

p0: 9.1740 +/- 1.2607
p1: 238.0708 +/- 4.3289
--------------------------------

--- fit Results for: Run Up 1; I [A]; B[mT] ---
Function: pol1
Chi2/NDF: 1.2877 / 11
p-value:  0.9998

p0: -8.6959 +/- 1.2339
p1: 238.1036 +/- 4.3465
--------------------------------

--- fit Results for: Run Down 2; I [A]; B[mT] ---
Function: pol1
Chi2/NDF: 1.3720 / 11
p-value:  0.9998

p0: 8.7601 +/- 1.2565
p1: 238.4401 +/- 4.3854
--------------------------------

--- fit Results for: Run Up 2; I [A]; B[mT] ---
Function: pol1
Chi2/NDF: 1.2914 / 11
p-value:  0.9998

p0: -8.6471 +/- 1.2327
p1: 239.1397 +/- 4.2969
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: MagneticFieldVsCurrent
Info in <TCanvas::Print>: png file outputs/MagneticFieldVsCurrent.png has been created


In [108]:
names = ["B_\\text{down,1}", "i_\\text{down,1}", "B_\\text{up,1}", "i_\\text{up,1}",
         "B_\\text{down,2}", "i_\\text{down,2}", "B_\\text{up,2}", "i_\\text{up,2}"]

data = [B1, i1, B2, i2, B3, i3, B4, i4]
errs = [errB1, erri1, errB2, erri2, errB3, erri3, errB4, erri4]

# appending elements (because some arrays are not long enough)
# then you can find and replace to remove the 77s

for i in range(4,6):
    data[i] = np.append(data[i], [[77,77,77,77,77]])
    errs[i] = np.append(errs[i], [[7,7,7,7,7]])

texTabler(data[0:4], errs[0:4], names[0:4])
texTabler(data[4:8], errs[4:8], names[4:8])

$B_\text{down,1}$ & $\delta B_\text{down,1}$ & $i_\text{down,1}$ & $\delta i_\text{down,1}$ & $B_\text{up,1}$ & $\delta B_\text{up,1}$ & $i_\text{up,1}$ & $\delta i_\text{up,1}$ \\
\hline
355 & 19 & 1.60 & 0.03 & -358 & 19 & -1.61 & 0.03\\
323 & 17 & 1.41 & 0.03 & -322 & 17 & -1.40 & 0.03\\
279 & 15 & 1.18 & 0.03 & -283 & 15 & -1.20 & 0.03\\
243 & 13 & 1.00 & 0.02 & -244 & 13 & -1.01 & 0.03\\
200 & 11 & 0.80 & 0.02 & -202 & 11 & -0.81 & 0.02\\
159 & 9 & 0.619 & 0.019 & -154 & 9 & -0.599 & 0.019\\
105 & 6 & 0.393 & 0.002 & -95 & 6 & -0.3530 & 0.0018\\
60 & 4 & 0.2080 & 0.0010 & -59 & 4 & -0.2060 & 0.0010\\
9.3 & 1.6 & 0.000000000 & 0.000000010 & -8.7 & 1.6 & 0.000000000 & 0.000000010\\
-48 & 3 & -0.2340 & 0.0012 & 40 & 3 & 0.2020 & 0.0010\\
-88 & 5 & -0.400 & 0.016 & 90 & 5 & 0.405 & 0.016\\
-136 & 8 & -0.603 & 0.019 & 139 & 8 & 0.611 & 0.019\\
-183 & 10 & -0.80 & 0.02 & 189 & 10 & 0.83 & 0.02\\
-229 & 12 & -1.01 & 0.03 & 232 & 13 & 1.03 & 0.03\\
-270 & 14 & -1.20 & 0.03 & 269 & 14 & 1.

NB: per i parametri ritornati dai fit, sto utilizzando il formato `np.array([[mean][error],...])` per scrivere i valori, così poi da avere le funzioni Zscore e MeanErrors per calcolare direttamente valori attessi ed errori

In [109]:
# verifico che i parametri in salita e discesa siano compatibili, poi medio tutto
# the uncertainty on q is estimated by the semi-difference (in both cases)

Zscore(param1, param3)
param_down = MeanError(param1, param3)

# reset error (but i dont agree with the metodology)
# param_down[0][1] = abs(param1[0][0] - param3[0][0]) / 2

print(f"down curves: B = ({param_down[0][0]:.3f} +- {param_down[0][1]:.3f})mT + I * ({param_down[1][0]:.3f} +- {param_down[1][1]:.3f}) mT/A \n")

Zscore(param2, param4)
param_up = MeanError(param2, param4)

# reset error (but i dont agree with the metodology)
# param_up[0][1] = abs(param2[0][0] - param4[0][0]) / 2

print(f"up curves: B = ({param_up[0][0]:.3f} +- {param_up[0][1]:.3f})mT + I * ({param_up[1][0]:.3f} +- {param_up[1][1]:.3f}) mT/A \n")

Zscore(param_up, param_down)
param = MeanError(param_up,param_down)

param[0][1] = abs(param_up[0][0] - param_down[0][0]) / 2

print(f"mean curve: B = ({param[0][0]:.3f} +- {param[0][1]:.3f})mT + I * ({param[1][0]:.3f} +- {param[1][1]:.3f}) mT/A \n")

z value of param 0 : -0.232
z value of param 1 : 0.0599


down curves: B = (8.966 +- 0.890)mT + I * (238.253 +- 3.081) mT/A 

z value of param 0 : 0.028
z value of param 1 : 0.17


up curves: B = (-8.671 +- 0.872)mT + I * (238.628 +- 3.056) mT/A 

z value of param 0 : 14.2
z value of param 1 : -0.0863


mean curve: B = (-0.032 +- 8.819)mT + I * (238.442 +- 2.170) mT/A 



In [110]:
# here we need to build the functions to translate current that goes in the electromagnet into magnetic field
# be mindful of the fact that we are using the named parameters in the cell above as default values

# UNITS: B in mT and I in A

def hysteresisUp(I, err_I, q=param_up[0][0], err_q=param_up[0][1], m=param_up[1][0], err_m=param_up[1][1]):
    """ returns  B, errB (mag field) np.arrays after taking I, errI (current in Amperes), for hystUP. optionally hysteresis curve parameters """
    B    = q + m * I
    errB = np.sqrt(err_q**2 + (I*err_m)**2 + (m*err_I)**2)

    return B, errB

def hysteresisDown(I, err_I, q=param_down[0][0], err_q=param_down[0][1], m=param_down[1][0], err_m=param_down[1][1]):
    """ returns  B, errB (mag field) np.arrays after taking I, errI (current), for hystDOWN. optionally hysteresis curve parameters """
    B    = q + m * I
    errB = np.sqrt(err_q**2 + (I*err_m)**2 + (m*err_I)**2)

    return B, errB

def hysteresisMean(I, err_I, q=param[0][0], err_q=param[0][1], m=param[1][0], err_m=param[1][1]):
    """ returns  B, errB (mag field) np.arrays after taking I, errI (current). optionally hysteresis curve parameters """
    B    = q + m * I
    errB = np.sqrt(err_q**2 + (I*err_m)**2 + (m*err_I)**2)

    return B, errB

## Tensione di Hall al variare di $i$

Vogliamo valutare l'andamento della tensione di Hall (sui lati del materiale semiconduttore) al variare della corrente iniettata al suo interno, in modo da stimare il parametro $R_H$.

Quindi fissato il valore del campo magnetico (una volta a zero, e poi a 5 valori diversi in entrambi i versi) valutiamo l'andamento $V_H(i)$ (*andando tra **-8mA e +8mA***) sapendo che in generale:
$$V_H = \frac{R_H}{t} \cdot i_p \cdot B$$
allora facciamo dei fit lineari su ogni set di dati e diamo una stima del parametro $R_H$ per entrambi i versi del campo magnetico, poi mediamo tra i due set.

In [111]:
# correction for misalignment of transverse contacts
# we will need to subtract it from the other fits

# UNITS: V in mV and I in mA

VH0    = np.array([-0.001, 0.055, 0.109, 0.165, 0.22, 0.276, 0.328, 0.384, 0.439, -0.051, -0.104, -0.159, -0.214, -0.267, -0.319, -0.374, -0.429])
errVH0 = Keithley(VH0)
Ip0    = np.array([0, 1.009, 1.999, 2.989, 4, 5.012, 5.977, 6.977, 7.997, -0.996, -1.986, -2.998, -4.003, -4.995, -5.975, -6.985, -8.009])
errIp0 = Amprobe(Ip0,unit="mA")


hallCurrPlotter = fitPlotter("HallTensionVsCurrent")
param0 = hallCurrPlotter.addGraph(Ip0, VH0, errIp0, errVH0, "Misalignment; i [mA]; V_{H} [mV]")

_ = hallCurrPlotter.addGraph(Ip0, VH0, errIp0, errVH0, "Misalignment new model; i [mA]; V_{H} [mV]",fit_formula="[0] * x")
_ = hallCurrPlotter.addGraph(Ip0, VH0, errIp0, errVH0, "Misalignment cut 1; i [mA]; V_{H} [mV]",xrange=[[-9,0],[0,9]],fit_formula=["pol1","pol1"])


hallCurrPlotter.drawCanvas(legend=False,dimX=500,dimY=250)
hallCurrPlotter.saveCanvas("outputs/HallTensionVsCurrent.png")


--- fit Results for: Misalignment; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 3.9827 / 15
p-value:  0.9978

p0: 0.0032 +/- 0.0009
p1: 0.0542 +/- 0.0002
--------------------------------

--- fit Results for: Misalignment new model; i [mA]; V_{H} [mV] ---
Function: [0] * x
Chi2/NDF: 15.8218 / 16
p-value:  0.4655

p0: 0.0542 +/- 0.0002
--------------------------------

--- fit Results for: Misalignment cut 1; i [mA]; V_{H} [mV] --- 
 ! Multiple fits are being committed !
fit 0, f = pol1
Function: pol1
Chi2/NDF: 0.7401 / 7
p-value:  0.9980

p0: 0.0012 +/- 0.0022
p1: 0.0537 +/- 0.0005
--------------------------------
fit 1, f = pol1
Function: pol1
Chi2/NDF: 0.1963 / 7
p-value:  1.0000

p0: -0.0006 +/- 0.0022
p1: 0.0551 +/- 0.0005
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallTensionVsCurrent
Info in <TCanvas::Print>: png file outputs/HallTensionVsCurrent.png has been created


In [112]:
from sigfig import round

def texTabler(data, errors, names):
    """ turns data = [np.array, np.array, ...] and errors = [np.array, np.array, ...] into table with names = [string, string, ...] """
    if len(data) != len(errors) or len(data) != len(names):
        raise IndexError("mistakes in lists lenghts! check them")

    title = ""
    for i in range(0,len(names)-1):
        title += f"${names[i]}$ & $\delta {names[i]}$ & "
    title+= f"${names[len(names)-1]}$ & $\delta {names[len(names)-1]}$ \\\\"

    print(title)
    print("\\hline")
    
    for i in range(0, len(data[0])):
        row = ""

        for j in range(0, len(data)-1):
            roundedString = round(data[j][i], errors[j][i], separation=' & ', cutoff=19)
            row += (roundedString) + " & "

        row += round(data[len(data)-1][i], errors[len(data) - 1][i], separation=' & ', cutoff=19) + "\\\\"

        print(row)

In [113]:
data = [VH0, Ip0]
err = [errVH0, errIp0]
names = ["V_{H}", "I_p"]

texTabler(data,err,names)

$V_{H}$ & $\delta V_{H}$ & $I_p$ & $\delta I_p$ \\
\hline
-0.001 & 0.004 & 0.000000 & 0.000010\\
0.055 & 0.004 & 1.009 & 0.010\\
0.109 & 0.004 & 1.999 & 0.015\\
0.165 & 0.004 & 2.99 & 0.02\\
0.220 & 0.004 & 4.00 & 0.03\\
0.276 & 0.004 & 5.01 & 0.03\\
0.328 & 0.004 & 5.98 & 0.03\\
0.384 & 0.004 & 6.98 & 0.04\\
0.439 & 0.004 & 8.00 & 0.04\\
-0.051 & 0.004 & -0.996 & 0.010\\
-0.104 & 0.004 & -1.986 & 0.015\\
-0.159 & 0.004 & -3.00 & 0.02\\
-0.214 & 0.004 & -4.00 & 0.03\\
-0.267 & 0.004 & -5.00 & 0.03\\
-0.319 & 0.004 & -5.98 & 0.03\\
-0.374 & 0.004 & -6.99 & 0.04\\
-0.429 & 0.004 & -8.01 & 0.05\\


In [114]:
# specifically we define the ohmic behaviour function to make corrections afterward (to the Vh(B) fits' results)

# UNITS: V in mV and I in mA

def ohmicContribution(I, errI,q=param0[0][0],errq=param0[0][1],m=param0[1][0],errm=param0[1][1]):
    """ returns Vh(I), errVh(I) for B=0 """
    Vh = q + m * I
    errVh = np.sqrt(errq**2 + (I*errm)**2 + (m*errI)**2)

    return Vh, errVh

# these are the corrections we will need to apply!
I = np.array([-2.002, -4.003, -6.005, 2.008, 4.000, 6.000])
errI = Amprobe(I, unit="mA")
ohmicContribution(I, errI)

(array([-0.10532297, -0.21377062, -0.32227247,  0.11200591,  0.21996579,
         0.32835925]),
 array([0.00129457, 0.00182589, 0.0024295 , 0.00129596, 0.00182502,
        0.00242794]))

In [115]:
# we are fitting Vh(I), using Amprobe and Keithley function to calculate the error

# UNITS: V in mV and I in mA

i1 = np.array([0, -0.995, -1.973, -2.989, -3.994, -4.988, -5.972, -7.002, -7.985, 1.001, 2.001, 3.019, 4, 4.993, 5.989, 7.002, 8.014]) #mA
VH1 = np.array([-0.01, -0.479, -0.938, -1.405, -1.875, -2.341, -2.801, -3.284, -3.743, 0.461, 0.93, 1.406, 1.866, 2.331, 2.798, 3.273, 3.747]) #mV
erri1 = Amprobe(i1, unit = "mA")
errVH1 = Keithley(VH1)

i2 = np.array([0, 0.988, 1.996, 2.989, 4, 4.998, 6.004, 6.959, 8.012, -1.011, -1.968, -3.005, -4.005, -5.052, -5.98, -7.033, -8.004]) #mA
VH2 = np.array([-0.008, 0.847, 1.721, 2.58, 3.455, 4.319, 5.189, 6.016, 6.927, -0.885, -1.714, -2.612, -3.477, -4.383, -5.185, -6.096, -6.936]) #mV
erri2 = Amprobe(i2, unit = "mA")
errVH2 = Keithley(VH2)

i3 = np.array([0, -1.001, -2.011, -3.029, -4.043, -4.983, -5.987, -6.999, -8.037, 1.002, 2.02, 2.995, 4.043, 5.003, 6.045, 7.012, 7.993]) #mA
VH3 = np.array([-0.009, -1.227, -2.452, -3.689, -4.919, -6.06, -7.279, -8.506, -9.765, 1.21, 2.445, 3.629, 4.899, 6.065, 7.329, 8.501, 9.691]) #mV
erri3 = Amprobe(i3, unit = "mA")
errVH3 = Keithley(VH3)

i4 = np.array([0, 1.025, 2.002, 3.038, 4.058, 5.032, 6.012, 7, 8.006, -1.002, -2.002, -2.998, -4.034, -4.992, -6.013, -7.015, -7.987]) #mA
VH4 = np.array([-0.008, 1.592, 3.113, 4.728, 6.317, 7.833, 9.358, 10.896, 12.461, -1.57, -3.127, -4.678, -6.291, -7.781, -9.368, -10.926, -12.436]) #mV
erri4 = Amprobe(i4, unit = "mA") 
errVH4 = Keithley(VH4)

i5 = np.array([0, -1.002, -2.002, -3.006, -3.992, -5.02, -6.003, -7.036, -8.021, 1.008, 1.988, 3, 3.997, 5.015, 5.962, 7.018, 8]) #mA
VH5 = np.array([-0.007, -1.891, -3.768, -5.653, -7.5, -9.415, -11.255, -13.189, -15.031, 1.884, 3.719, 5.614, 7.477, 9.381, 11.151, 13.127, 14.96]) #mV
erri5 = Amprobe(i5, unit = "mA")
errVH5 = Keithley(VH5)

hallVoltagePlotter = fitPlotter("HallVoltageVsCurrentBpos")


# from these parameters we need to remove the ohmic contribution
param1pos = hallVoltagePlotter.addGraph(i1,VH1,erri1,errVH1, title="Magnet Current 0.200 A; i [mA]; V_{H} [mV]")
param2pos = hallVoltagePlotter.addGraph(i2,VH2,erri2,errVH2, title="Magnet Current 0.425 A; i [mA]; V_{H} [mV]")
param3pos = hallVoltagePlotter.addGraph(i3,VH3,erri3,errVH3, title="Magnet Current 0.616 A; i [mA]; V_{H} [mV]")
param4pos = hallVoltagePlotter.addGraph(i4,VH4,erri4,errVH4, title="Magnet Current 0.811 A; i [mA]; V_{H} [mV]")
param5pos = hallVoltagePlotter.addGraph(i5,VH5,erri5,errVH5, title="Magnet Current 1.005 A; i [mA]; V_{H} [mV]")

hallVoltagePlotter.drawCanvas(legend=False)
hallVoltagePlotter.saveCanvas("outputs/hallVoltageVsCurrentBpos.png")


--- fit Results for: Magnet Current 0.200 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 1.7161 / 15
p-value:  1.0000

p0: -0.0091 +/- 0.0021
p1: 0.4683 +/- 0.0008
--------------------------------

--- fit Results for: Magnet Current 0.425 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.0760 / 15
p-value:  1.0000

p0: -0.0083 +/- 0.0027
p1: 0.8659 +/- 0.0014
--------------------------------

--- fit Results for: Magnet Current 0.616 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.1175 / 15
p-value:  1.0000

p0: -0.0091 +/- 0.0030
p1: 1.2143 +/- 0.0020
--------------------------------

--- fit Results for: Magnet Current 0.811 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.1560 / 15
p-value:  1.0000

p0: -0.0077 +/- 0.0032
p1: 1.5577 +/- 0.0025
--------------------------------

--- fit Results for: Magnet Current 1.005 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.6287 / 15
p-value:  1.0000

p0: -0.0076 +/- 0.0033
p1: 1.8738 +/- 0.0030
-----------------------

Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallVoltageVsCurrentBpos
Info in <TCanvas::Print>: png file outputs/hallVoltageVsCurrentBpos.png has been created


In [116]:
data1 = [VH1, i1, VH2, i2, VH3]
data2 = [i3, VH4, i4, VH5, i5]
err1  = [errVH1, erri1, errVH2, erri2, errVH3]
err2  = [erri3, errVH4, erri4, errVH5, erri5]


name1 = ["V_{H,1}", "i_{p,1}", "V_{H,2}", "i_{p,2}", "V_{H,3}"]
name2 = ["i_{p,3}", "V_{H,4}", "i_{p,4}", "V_{H,5}", "i_{p,5}"]

texTabler(data1,err1,name1)
texTabler(data2,err2,name2)

$V_{H,1}$ & $\delta V_{H,1}$ & $i_{p,1}$ & $\delta i_{p,1}$ & $V_{H,2}$ & $\delta V_{H,2}$ & $i_{p,2}$ & $\delta i_{p,2}$ & $V_{H,3}$ & $\delta V_{H,3}$ \\
\hline
-0.010 & 0.004 & 0.000000 & 0.000010 & -0.008 & 0.004 & 0.000000 & 0.000010 & -0.009 & 0.004\\
-0.479 & 0.004 & -0.995 & 0.010 & 0.847 & 0.004 & 0.988 & 0.010 & -1.227 & 0.004\\
-0.938 & 0.004 & -1.973 & 0.015 & 1.721 & 0.004 & 1.996 & 0.015 & -2.452 & 0.004\\
-1.405 & 0.004 & -2.99 & 0.02 & 2.580 & 0.004 & 2.99 & 0.02 & -3.689 & 0.004\\
-1.875 & 0.004 & -3.99 & 0.02 & 3.455 & 0.004 & 4.00 & 0.03 & -4.919 & 0.004\\
-2.341 & 0.004 & -4.99 & 0.03 & 4.319 & 0.004 & 5.00 & 0.03 & -6.060 & 0.004\\
-2.801 & 0.004 & -5.97 & 0.03 & 5.189 & 0.004 & 6.00 & 0.04 & -7.279 & 0.004\\
-3.284 & 0.004 & -7.00 & 0.04 & 6.016 & 0.004 & 6.96 & 0.04 & -8.506 & 0.004\\
-3.743 & 0.004 & -7.99 & 0.04 & 6.927 & 0.004 & 8.01 & 0.05 & -9.765 & 0.004\\
0.461 & 0.004 & 1.001 & 0.010 & -0.885 & 0.004 & -1.011 & 0.010 & 1.210 & 0.004\\
0.930 & 0.004 & 2.00

/var/folders/1c/z7ldvl9s3q52971d3_jc4zf40000gn/T/ipykernel_3644/801023698.py:20: UserWarning: 2 significant figures requested from number with only 1 significant figures
  roundedString = round(data[j][i], errors[j][i], separation=' & ', cutoff=19)
/var/folders/1c/z7ldvl9s3q52971d3_jc4zf40000gn/T/ipykernel_3644/801023698.py:23: UserWarning: 2 significant figures requested from number with only 1 significant figures
  row += round(data[len(data)-1][i], errors[len(data) - 1][i], separation=' & ', cutoff=19) + "\\\\"


In [117]:
# same mesaures but we inverted the magnetic field
# we still need to implement the errors on Hall tension
# we are fitting VH(I), using Amprobe and Keithley function to calculate the error

# UNITS: V in mV and I in mA

i1 = np.array([0, 1.007, 1.988, 2.991, 4.011, 5, 5.988, 7.009, 8.001, -1.002, -1.999, -3.006, -3.997, -5.039, -6.014, -6.953, -8.07])
VH1 = np.array([-0.002, -0.267, -0.525, -0.788, -1.056, -1.316, -1.575, -1.843, -2.104, 0.261, 0.522, 0.786, 1.046, 1.32, 1.576, 1.822, 2.115])
erri1 = Amprobe(i1, unit = "mA")
errVH1 = Keithley(VH1)

i2 = np.array([0, -1.025, -1.902, -2.997, -4.049, -4.955, -6.075, -7.068, -7.974, 0.922, 2.003, 3.02, 4.006, 5.021, 5.953, 6.998, 7.978])
VH2 = np.array([-0.002, 0.684, 1.267, 2.001, 2.704, 3.309, 4.057, 4.719, 5.325, -0.618, -1.345, -2.027, -2.688, -3.369, -3.995, -4.696, -5.355])
erri2 = Amprobe(i2, unit = "mA")
errVH2 = Keithley(VH2)

i3 = np.array([0, 1.011, 1.934, 2.984, 3.94, 4.944, 5.988, 7.042, 8.029, -0.985, -1.912, -3.007, -4.036, -4.945, -5.925, -6.98, -7.992])
VH3 = np.array([-0.001, -1.056, -2.021, -3.114, -4.112, -5.159, -6.248, -7.347, -8.377, 1.028, 1.997, 3.139, 4.211, 5.16, 6.183, 7.283, 8.338])
erri3 = Amprobe(i3, unit = "mA")
errVH3 = Keithley(VH3)

# in this dataset there's obviously a mistake (maybe we wrote a wrong value: 4.001, -5.268; index = 12)
i4 = np.array([0, -1.006, -2.016, -2.972, -3.973, -5.01, -5.966, -6.996, -7.97, 0.975, 1.993, 2.997, 4.001, 4.983, 6.004, 6.978, 7.984])
VH4 = np.array([0, 1.419, 2.838, 4.186, 5.594, 7.053, 8.398, 9.846, 11.216, -1.372, -2.806, -4.217, -5.268, -7.011, -8.446, -9.814, -11.227])
erri4 = Amprobe(i4, unit = "mA")
errVH4 = Keithley(VH4)

# removing problematic data point
i4     = np.delete(i4, 12)
VH4    = np.delete(VH4, 12)
erri4  = np.delete(erri4, 12)
errVH4 = np.delete(errVH4, 12);

i5 = np.array([0, 0.97, 1.998, 2.983, 4.002, 5.001, 6.009, 6.997, 7.971, -0.997, -2.007, -2.996, -3.987, -5.011, -6, -7.007, -8.013])
VH5 = np.array([0, -1.665, -3.415, -5.122, -6.867, -8.58, -10.308, -12.001, -13.668, 1.713, 3.443, 5.139, 6.847, 8.586, 10.28, 12.003, 13.728])
erri5 = Amprobe(i5, unit = "mA")
errVH5 = Keithley(VH5)

hallVoltagePlotter = fitPlotter("HallVoltageVsCurrentBneg")

# from these parameters we need to remove the ohmic contribution
param1neg = hallVoltagePlotter.addGraph(i1,VH1,erri1,errVH1, title="Magnet Current -0.200 A; i [mA]; V_{H} [mV]")
param2neg = hallVoltagePlotter.addGraph(i2,VH2,erri2,errVH2, title="Magnet Current -0.409 A; i [mA]; V_{H} [mV]]")
param3neg = hallVoltagePlotter.addGraph(i3,VH3,erri3,errVH3, title="Magnet Current -0.606 A; i [mA]; V_{H} [mV]")
param4neg = hallVoltagePlotter.addGraph(i4,VH4,erri4,errVH4, title="Magnet Current -0.813 A; i [mA]; V_{H} [mV]")
param5neg = hallVoltagePlotter.addGraph(i5,VH5,erri5,errVH5, title="Magnet Current -0.999 A; i [mA]; V_{H} [mV]")

hallVoltagePlotter.drawCanvas(legend=False,statX=0.9,statY=0.8)
hallVoltagePlotter.saveCanvas("outputs/hallVoltageVsCurrentBneg.png")


--- fit Results for: Magnet Current -0.200 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.1025 / 15
p-value:  1.0000

p0: -0.0027 +/- 0.0015
p1: -0.2625 +/- 0.0005
--------------------------------

--- fit Results for: Magnet Current -0.409 A; i [mA]; V_{H} [mV]] ---
Function: pol1
Chi2/NDF: 1.1588 / 15
p-value:  1.0000

p0: -0.0033 +/- 0.0025
p1: -0.6694 +/- 0.0011
--------------------------------

--- fit Results for: Magnet Current -0.606 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.0710 / 15
p-value:  1.0000

p0: -0.0008 +/- 0.0029
p1: -1.0436 +/- 0.0017
--------------------------------

--- fit Results for: Magnet Current -0.813 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.1101 / 14
p-value:  1.0000

p0: 0.0003 +/- 0.0031
p1: -1.4074 +/- 0.0023
--------------------------------

--- fit Results for: Magnet Current -0.999 A; i [mA]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 0.4502 / 15
p-value:  1.0000

p0: 0.0001 +/- 0.0032
p1: -1.7148 +/- 0.0028
--------------

Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallVoltageVsCurrentBneg
Info in <TCanvas::Print>: png file outputs/hallVoltageVsCurrentBneg.png has been created


In [118]:
data1 = [VH1, i1, VH2, i2, VH3]
data2 = [i3, np.append(VH4, [77]), np.append(i4, [77]), VH5, i5]
err1  = [errVH1, erri1, errVH2, erri2, errVH3]
err2  = [erri3, np.append(errVH4, [7]), np.append(erri4, [7]), errVH5, erri5]
# we append some stuff to uniformise lenghts after problematic data removal

name1 = ["V_{H,1}", "i_{p,1}", "V_{H,2}", "i_{p,2}", "V_{H,3}"]
name2 = ["i_{p,3}", "V_{H,4}", "i_{p,4}", "V_{H,5}", "i_{p,5}"]

texTabler(data1,err1,name1)
texTabler(data2,err2,name2)

$V_{H,1}$ & $\delta V_{H,1}$ & $i_{p,1}$ & $\delta i_{p,1}$ & $V_{H,2}$ & $\delta V_{H,2}$ & $i_{p,2}$ & $\delta i_{p,2}$ & $V_{H,3}$ & $\delta V_{H,3}$ \\
\hline
-0.002 & 0.004 & 0.000000 & 0.000010 & -0.002 & 0.004 & 0.000000 & 0.000010 & -0.001 & 0.004\\
-0.267 & 0.004 & 1.007 & 0.010 & 0.684 & 0.004 & -1.025 & 0.010 & -1.056 & 0.004\\
-0.525 & 0.004 & 1.988 & 0.015 & 1.267 & 0.004 & -1.902 & 0.015 & -2.021 & 0.004\\
-0.788 & 0.004 & 2.99 & 0.02 & 2.001 & 0.004 & -3.00 & 0.02 & -3.114 & 0.004\\
-1.056 & 0.004 & 4.01 & 0.03 & 2.704 & 0.004 & -4.05 & 0.03 & -4.112 & 0.004\\
-1.316 & 0.004 & 5.00 & 0.03 & 3.309 & 0.004 & -4.96 & 0.03 & -5.159 & 0.004\\
-1.575 & 0.004 & 5.99 & 0.03 & 4.057 & 0.004 & -6.08 & 0.04 & -6.248 & 0.004\\
-1.843 & 0.004 & 7.01 & 0.04 & 4.719 & 0.004 & -7.07 & 0.04 & -7.347 & 0.004\\
-2.104 & 0.004 & 8.00 & 0.05 & 5.325 & 0.004 & -7.97 & 0.04 & -8.377 & 0.004\\
0.261 & 0.004 & -1.002 & 0.010 & -0.618 & 0.004 & 0.922 & 0.010 & 1.028 & 0.004\\
0.522 & 0.004 & -1.9

In [119]:
# here the parameters are stored as [[q, qerr], [m, merr]]
# sto sottraendo i parametri della caratteristica V_H(i) con B=0 a V_H(i) a B variabile

# UNITS: m in mV/mA = V/A and q in mV

parampos = [param1pos, param2pos, param3pos, param4pos, param5pos]
paramneg = [param1neg, param2neg, param3neg, param4neg, param5neg]

m     = []
err_m = []

slopes = []
err_slopes = []

q     = []
err_q = []

quot = []
err_quot = []

for pos in parampos:
    q.append(pos[0][0] - param0[0][0])
    err_q.append(np.sqrt(pos[0][1]**2 + param0[0][1]**2))

    quot.append(pos[0][0])
    err_quot.append(pos[0][1])

    slopes.append(pos[1][0])
    err_slopes.append(pos[1][1])

    m.append(pos[1][0] - param0[1][0])
    err_m.append(np.sqrt(pos[1][1]**2 + param0[1][1]**2))

for pos in paramneg:
    q.append(pos[0][0] - param0[0][0])
    err_q.append(np.sqrt(pos[0][1]**2 + param0[0][1]**2))

    quot.append(pos[0][0])
    err_quot.append(pos[0][1])

    slopes.append(pos[1][0])
    err_slopes.append(pos[1][1])
    
    m.append(pos[1][0] - param0[1][0])
    err_m.append(np.sqrt(pos[1][1]**2 + param0[1][1]**2))

m = np.array(m)
err_m = np.array(err_m)

q = np.array(q)
err_q = np.array(err_q)


print(q)
print(err_q)

[-0.01232885 -0.01150415 -0.01226067 -0.01089274 -0.01073404 -0.00584913
 -0.00645334 -0.0039474  -0.00287314 -0.0030758 ]
[0.00227085 0.0028819  0.00316214 0.00331412 0.00339618 0.00179835
 0.00262304 0.00303978 0.00326635 0.00335628]


In [120]:
Ineg = np.array([-0.200,-0.409,-0.606,-0.813, -0.999])
errIneg = Amprobe(Ineg, unit="A")
Ipos = np.array([0.200, 0.425, 0.616, 0.811, 1.005])
errIpos = Amprobe(Ipos, unit="A")
Iposneg = np.concatenate((Ipos, Ineg),axis=None)
errIposneg = np.concatenate((errIpos, errIneg), axis=None)

Bneg, errBneg = hysteresisMean(Ineg, errIneg)
Bpos, errBpos = hysteresisMean(Ipos, errIpos)
Bposneg    = np.concatenate((Bpos, Bneg), axis=None)
errBposneg = np.concatenate((errBpos, errBneg), axis=None)

data = [np.array(slopes), np.array(quot), m, q, Iposneg, Bposneg]
err  = [np.array(err_slopes), np.array(err_quot), err_m, err_q, errIposneg, errBposneg]
lab  = ["m", "q", "m_\\text{corr}", "q_\\text{corr}", "I", "B"]

texTabler(data, err, lab)

$m$ & $\delta m$ & $q$ & $\delta q$ & $m_\text{corr}$ & $\delta m_\text{corr}$ & $q_\text{corr}$ & $\delta q_\text{corr}$ & $I$ & $\delta I$ & $B$ & $\delta B$ \\
\hline
0.4683 & 0.0008 & -0.009 & 0.002 & 0.4141 & 0.0008 & -0.012 & 0.002 & 0.2000 & 0.0010 & 48 & 9\\
0.8659 & 0.0014 & -0.008 & 0.003 & 0.8117 & 0.0014 & -0.012 & 0.003 & 0.425 & 0.016 & 101 & 10\\
1.214 & 0.002 & -0.009 & 0.003 & 1.160 & 0.002 & -0.012 & 0.003 & 0.616 & 0.019 & 147 & 10\\
1.558 & 0.003 & -0.008 & 0.003 & 1.503 & 0.003 & -0.011 & 0.003 & 0.81 & 0.02 & 193 & 10\\
1.874 & 0.003 & -0.008 & 0.003 & 1.820 & 0.003 & -0.011 & 0.003 & 1.01 & 0.03 & 240 & 11\\
-0.2625 & 0.0005 & -0.0027 & 0.0015 & -0.3167 & 0.0005 & -0.0058 & 0.0018 & -0.2000 & 0.0010 & -48 & 9\\
-0.6694 & 0.0011 & -0.003 & 0.002 & -0.7235 & 0.0011 & -0.006 & 0.003 & -0.409 & 0.016 & -98 & 10\\
-1.0436 & 0.0017 & -0.001 & 0.003 & -1.0978 & 0.0017 & -0.004 & 0.003 & -0.606 & 0.019 & -145 & 10\\
-1.407 & 0.002 & 0.000 & 0.003 & -1.462 & 0.002 & -0.00

### **Test Z compatibilità con 0 delle quote dei fit**

A questo punto effettuiamo dei test Z per verificare la compatibilità con una distribuzione normale dei parametri quota: non sono tutti compatibili!

In [121]:
# it's a reason to look more closely at the errors (by flipping the sign of the correction the situation improves: think about it)
s1 = np.column_stack((q, err_q))
s2 = np.zeros_like(s1)
z_scores = Zscore(s1, s2)
errz = np.ones(len(z_scores)) *0.002


data = [Bposneg, np.array(quot), q, z_scores]
errs = [errBposneg, np.array(err_quot), err_q, errz]
labs = ["B", "q", "q_\\text{corr}", "Z"]

texTabler(data, errs, labs)

z value of param 0 : 5.43
z value of param 1 : 3.99
z value of param 2 : 3.88
z value of param 3 : 3.29
z value of param 4 : 3.16
z value of param 5 : 3.25
z value of param 6 : 2.46
z value of param 7 : 1.3
z value of param 8 : 0.88
z value of param 9 : 0.916


$B$ & $\delta B$ & $q$ & $\delta q$ & $q_\text{corr}$ & $\delta q_\text{corr}$ & $Z$ & $\delta Z$ \\
\hline
48 & 9 & -0.009 & 0.002 & -0.012 & 0.002 & 5.429 & 0.002\\
101 & 10 & -0.008 & 0.003 & -0.012 & 0.003 & 3.992 & 0.002\\
147 & 10 & -0.009 & 0.003 & -0.012 & 0.003 & 3.877 & 0.002\\
193 & 10 & -0.008 & 0.003 & -0.011 & 0.003 & 3.287 & 0.002\\
240 & 11 & -0.008 & 0.003 & -0.011 & 0.003 & 3.161 & 0.002\\
-48 & 9 & -0.0027 & 0.0015 & -0.0058 & 0.0018 & 3.253 & 0.002\\
-98 & 10 & -0.003 & 0.002 & -0.006 & 0.003 & 2.460 & 0.002\\
-145 & 10 & -0.001 & 0.003 & -0.004 & 0.003 & 1.299 & 0.002\\
-194 & 10 & 0.000 & 0.003 & -0.003 & 0.003 & 0.880 & 0.002\\
-238 & 11 & 0.000 & 0.003 & -0.003 & 0.003 & 0.916 & 0.002\\


/var/folders/1c/z7ldvl9s3q52971d3_jc4zf40000gn/T/ipykernel_3644/801023698.py:23: UserWarning: 2 significant figures requested from number with only 1 significant figures
  row += round(data[len(data)-1][i], errors[len(data) - 1][i], separation=' & ', cutoff=19) + "\\\\"


In [122]:
correctionPlotter = fitPlotter("correction for intercepts")
_ = correctionPlotter.addGraph(Bposneg, np.array(quot), errBposneg, np.array(err_quot), fit_formula=None, title="q against B; B [mT]; q [mV]")
correctionPlotter.drawCanvas(legend=False)
correctionPlotter.saveCanvas("outputs/intercepts.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: correction for intercepts
Info in <TCanvas::Print>: png file outputs/intercepts.png has been created


### **Stima di $R_h$**

In [123]:
# HERE WE PERFORM LINEAR FIT of M (slope of V_H(i) curves against B) to get R_H!

# UNITS: I in A, B in mT, m in V/A => R_H in (V/A) * mm / mT = m^3 / C

Ineg = np.array([-0.200,-0.409,-0.606,-0.813, -0.999])
errIneg = Amprobe(Ineg, unit="A")

Ipos = np.array([0.200, 0.425, 0.616, 0.811, 1.005])
errIpos = Amprobe(Ipos, unit="A")

Bneg, errBneg = hysteresisMean(Ineg, errIneg)
Bpos, errBpos = hysteresisMean(Ipos, errIpos)

# this is the order in which i get the slopes: parampos then paramneg
Bposneg    = np.concatenate((Bpos, Bneg), axis=None)
errBposneg = np.concatenate((errBpos, errBneg), axis=None)

RHfromVhVsIplotter = fitPlotter("SlopesVhIVsB")

# where coeff = R_H / t
[stuff, [coeff, err_coeff]] = RHfromVhVsIplotter.addGraph(Bposneg, m, errBposneg, err_m, title="(corrected) Slopes against B; B [mT]; m [V/A]")
RHfromVhVsIplotter.drawCanvas(legend=False)
RHfromVhVsIplotter.saveCanvas("outputs/slopesAgainstBtogetRH.png")

# here we determine R_H
t = 1
err_t = 0.1 # need to check: it's a big error...

R_H1    = coeff * t
errR_H1 = np.sqrt((coeff*err_t)**2 + (t*err_coeff)**2)

rho1 = 1 / R_H1
err_rho1 = (errR_H1 / R_H1) * rho1

print(f"R_H = {R_H1} +- {errR_H1} [m^3 / C]")
print(f"rho = {rho1} +- {err_rho1} [C / m^3]")


--- fit Results for: (corrected) Slopes against B; B [mT]; m [V/A] ---
Function: pol1
Chi2/NDF: 0.5992 / 8
p-value:  0.9997

p0: 0.0303 +/- 0.0238
p1: 0.0076 +/- 0.0002
--------------------------------
R_H = 0.007620807231447561 +- 0.0007781605188489423 [m^3 / C]
rho = 131.21969492594704 +- 13.398841196955354 [C / m^3]


Warning in <TCanvas::Constructor>: Deleting canvas with same name: SlopesVhIVsB
Info in <TCanvas::Print>: png file outputs/slopesAgainstBtogetRH.png has been created


## Tensione di Hall al variare di $B$

Ripetiamo sostanzialmente le misure al punto precedente, ma invertendo i ruoli di variabile dipendente e indipendente. Adesso fissiamo $i_p$, scegliendo $3 \times 2$ valori, (dopo aver tracciato un'altra curva di caduta di potenziale ohmica a $B=0$) e studiamo il variare di $V_H$ con $B$.

In [124]:
# same thing as before but now we are changing the magnetic field
# BE MINDFUL: some data are taken in saturation... not good!

# here error propagation on currents and tensions with multimeters

# UNITS: I in A, V in mV, and B in mT

IB1 = np.array([-1.195, -0.88, -0.602, -0.287, 0, 0.41, 0.65, 0.902, 1.178])
errIB1 = Amprobe(IB1, unit="A")
VH1 = np.array([4.319, 3.483, 2.554, 1.371, 0.236, -1.376, -2.276, -3.148, -4.014])
errVH1 = Keithley(VH1)

IB2 = np.array([1.177, 0.711, 0.552, 0.304, 0, -0.315, -0.597, -0.978, -1.238])
errIB2 = Amprobe(IB2, unit="A")
VH2 = np.array([-7.997, -5.371, -4.271, -2.413, -0.018, 2.468, 4.605, 7.252, 8.844])
errVH2 = Keithley(VH2)

IB3 = np.array([-1.237, -0.894, -0.609, -0.334, 0, 0.342, 0.587, 0.915, 1.189])
errIB3 = Amprobe(IB3, unit="A")
VH3 = np.array([13.258, 10.537, 7.704, 4.642, 0.705, -3.335, -6.106, -9.544, -12.09])
errVH3 = Keithley(VH3)


# now we translate currents in electromagnet to mag fields with specific hysteresis

B1, errB1 = hysteresisUp(IB1, errIB1)
B2, errB2 = hysteresisDown(IB2, errIB2)
B3, errB3 = hysteresisUp(IB3, errIB3)

# inverting sign of B* (we inverted signs in data-taking)
B1, B2, B3 = -B1, -B2, -B3

hallVoltageBFieldPlotter = fitPlotter("HallVoltageVsMagneticFieldIpos")

param1 = hallVoltageBFieldPlotter.addGraph(B1,VH1,errB1,errVH1, title="i probe 2 mA; B [mT]; V_{H} [mV]", xrange = [-250,+250])
param2 = hallVoltageBFieldPlotter.addGraph(B2,VH2,errB2,errVH2, title="i probe 4 mA; B [mT]; V_{H} [mV]", xrange = [-250,+250])
param3 = hallVoltageBFieldPlotter.addGraph(B3,VH3,errB3,errVH3, title="i probe 6 mA; B [mT]; V_{H} [mV]", xrange = [-250,+250])

hallVoltageBFieldPlotter.drawCanvas(legend=False)
hallVoltageBFieldPlotter.saveCanvas("outputs/hallVoltageVsBFieldIpos.png")


iPos = np.array([2.008, 4.000, 6.000])
interPos = np.array([param1[0], param2[0], param3[0]])
coeffPos = np.array([param1[1], param2[1], param3[1]])


--- fit Results for: i probe 2 mA; B [mT]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 4.6635 / 5
p-value:  0.4583

p0: 0.1029 +/- 0.0125
p1: 0.0161 +/- 0.0002
--------------------------------

--- fit Results for: i probe 4 mA; B [mT]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 4.5785 / 5
p-value:  0.4695

p0: 0.2750 +/- 0.0210
p1: 0.0326 +/- 0.0003
--------------------------------

--- fit Results for: i probe 6 mA; B [mT]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 6.4855 / 5
p-value:  0.2618

p0: 0.2783 +/- 0.0314
p1: 0.0487 +/- 0.0005
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallVoltageVsMagneticFieldIpos
Info in <TCanvas::Print>: png file outputs/hallVoltageVsBFieldIpos.png has been created


In [125]:
data = [VH1, IB1, B1, VH2, IB2, B2, VH3, IB3, B3]
errs = [errVH1, errIB1, errB1, errVH2, errIB2, errB2, errVH3, errIB3, errB3]
lab  = ["V_{H,1}", "I_1", "B_1", "V_{H,2}", "I_2", "B_2", "V_{H,3}", "I_3", "B_3"]

texTabler(data[0:5], errs[0:5], lab[0:5])
texTabler(data[5:10], errs[5:10], lab[5:10])

$V_{H,1}$ & $\delta V_{H,1}$ & $I_1$ & $\delta I_1$ & $B_1$ & $\delta B_1$ & $V_{H,2}$ & $\delta V_{H,2}$ & $I_2$ & $\delta I_2$ \\
\hline
4.319 & 0.004 & -1.20 & 0.03 & 294 & 8 & -7.997 & 0.004 & 1.18 & 0.03\\
3.483 & 0.004 & -0.88 & 0.02 & 219 & 6 & -5.371 & 0.004 & 0.71 & 0.02\\
2.554 & 0.004 & -0.602 & 0.019 & 152 & 5 & -4.271 & 0.004 & 0.552 & 0.018\\
1.371 & 0.004 & -0.2870 & 0.0014 & 77.2 & 1.3 & -2.413 & 0.004 & 0.3040 & 0.0015\\
0.236 & 0.004 & 0.000000000 & 0.000000010 & 8.7 & 0.9 & -0.018 & 0.004 & 0.000000000 & 0.000000010\\
-1.376 & 0.004 & 0.410 & 0.016 & -89 & 4 & 2.468 & 0.004 & -0.3150 & 0.0016\\
-2.276 & 0.004 & 0.65 & 0.02 & -146 & 5 & 4.605 & 0.004 & -0.597 & 0.019\\
-3.148 & 0.004 & 0.90 & 0.02 & -207 & 6 & 7.252 & 0.004 & -0.98 & 0.02\\
-4.014 & 0.004 & 1.18 & 0.03 & -272 & 8 & 8.844 & 0.004 & -1.24 & 0.03\\
$B_2$ & $\delta B_2$ & $V_{H,3}$ & $\delta V_{H,3}$ & $I_3$ & $\delta I_3$ & $B_3$ & $\delta B_3$ \\
\hline
-289 & 8 & 13.258 & 0.004 & -1.24 & 0.03 & 304 & 8

In [126]:
# now the current flows the other way

# UNITS: I in A, V in mV, and B in mT

IB1 = np.array([1.207, 0.9, 0.546, 0.277, 0, -0.3, -0.497, -0.918, -1.211])
errIB1 = Amprobe(IB1, unit="A")
VH1 = np.array([4.093, 3.307, 2.132, 1.118, 0.025, -1.16, -1.918, -3.426, -4.348])
errVH1 = Keithley(VH1)

IB2 = np.array([-1.211, -0.907, -0.602, -0.312, 0, 0.303, 0.725, 0.897, 1.225])
errIB2 = Amprobe(IB2, unit="A")
VH2 = np.array([-8.697, -7.102, -5.088, -2.919, -0.462, 1.935, 5.078, 6.252, 8.284])
errVH2 = Keithley(VH2)

IB3 = np.array([1.221, 0.879, 0.604, 0.321, 0, -0.4, -0.638, -0.928, -1.197])
errIB3 = Amprobe(IB3, unit="A")
VH3 = np.array([12.403, 9.723, 6.992, 3.838, 0.048, -4.667, -7.339, -10.372, -12.901])
errVH3 = Keithley(VH3)

# now we translate currents in electromagnet to mag fields with specific hysteresis

B1, errB1 = hysteresisDown(IB1, errIB1)
B2, errB2 = hysteresisUp(IB2, errIB2)
B3, errB3 = hysteresisDown(IB3, errIB3)

# inverting sign of B* (we inverted signs in data-taking)

B1, B2, B3 = -B1, -B2, -B3


hallVoltageBFieldPlotter = fitPlotter("HallVoltageVsMagneticFieldNeg")

param1 = hallVoltageBFieldPlotter.addGraph(B1,VH1,errB1,errVH1,title="i probe -2 mA; B [mT]; V_{H} [mV]", xrange = [-250,+250])
param2 = hallVoltageBFieldPlotter.addGraph(B2,VH2,errB2,errVH2,title="i probe -4 mA; B [mT]; V_{H} [mV]", xrange = [-250,+250])
param3 = hallVoltageBFieldPlotter.addGraph(B3,VH3,errB3,errVH3,title="i probe -6 mA; B [mT]; V_{H} [mV]", xrange = [-250,+250])

hallVoltageBFieldPlotter.drawCanvas(legend=False,statX=0.9, statY=0.85)
hallVoltageBFieldPlotter.saveCanvas("outputs/hallVoltageVsBFieldNeg.png")

iNeg = np.array([-2.002, -4.003, -6.005])
interNeg = np.array([param1[0], param2[0], param3[0]])
coeffNeg = np.array([param1[1], param2[1], param3[1]])


--- fit Results for: i probe -2 mA; B [mT]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 6.9934 / 5
p-value:  0.2211

p0: -0.1228 +/- 0.0104
p1: -0.0163 +/- 0.0002
--------------------------------

--- fit Results for: i probe -4 mA; B [mT]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 7.2161 / 5
p-value:  0.2051

p0: -0.1770 +/- 0.0206
p1: -0.0325 +/- 0.0003
--------------------------------

--- fit Results for: i probe -6 mA; B [mT]; V_{H} [mV] ---
Function: pol1
Chi2/NDF: 4.9504 / 5
p-value:  0.4220

p0: -0.3686 +/- 0.0378
p1: -0.0484 +/- 0.0006
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallVoltageVsMagneticFieldNeg
Info in <TCanvas::Print>: png file outputs/hallVoltageVsBFieldNeg.png has been created


In [127]:
data = [VH1, IB1, B1, VH2, IB2, B2, VH3, IB3, B3]
errs = [errVH1, errIB1, errB1, errVH2, errIB2, errB2, errVH3, errIB3, errB3]
lab  = ["V_{H,1}", "I_1", "B_1", "V_{H,2}", "I_2", "B_2", "V_{H,3}", "I_3", "B_3"]

texTabler(data[0:5], errs[0:5], lab[0:5])
texTabler(data[5:10], errs[5:10], lab[5:10])

$V_{H,1}$ & $\delta V_{H,1}$ & $I_1$ & $\delta I_1$ & $B_1$ & $\delta B_1$ & $V_{H,2}$ & $\delta V_{H,2}$ & $I_2$ & $\delta I_2$ \\
\hline
4.093 & 0.004 & 1.21 & 0.03 & -297 & 8 & -8.697 & 0.004 & -1.21 & 0.03\\
3.307 & 0.004 & 0.90 & 0.02 & -223 & 6 & -7.102 & 0.004 & -0.91 & 0.02\\
2.132 & 0.004 & 0.546 & 0.018 & -139 & 5 & -5.088 & 0.004 & -0.602 & 0.019\\
1.118 & 0.004 & 0.2770 & 0.0014 & -75.0 & 1.3 & -2.919 & 0.004 & -0.3120 & 0.0016\\
0.025 & 0.004 & 0.000000000 & 0.000000010 & -9.0 & 0.9 & -0.462 & 0.004 & 0.000000000 & 0.000000010\\
-1.160 & 0.004 & -0.3000 & 0.0015 & 62.5 & 1.3 & 1.935 & 0.004 & 0.3030 & 0.0015\\
-1.918 & 0.004 & -0.497 & 0.017 & 109 & 5 & 5.078 & 0.004 & 0.73 & 0.02\\
-3.426 & 0.004 & -0.92 & 0.02 & 210 & 6 & 6.252 & 0.004 & 0.90 & 0.02\\
-4.348 & 0.004 & -1.21 & 0.03 & 280 & 8 & 8.284 & 0.004 & 1.23 & 0.03\\
$B_2$ & $\delta B_2$ & $V_{H,3}$ & $\delta V_{H,3}$ & $I_3$ & $\delta I_3$ & $B_3$ & $\delta B_3$ \\
\hline
298 & 8 & 12.403 & 0.004 & 1.22 & 0.03 & -3

In [128]:
# UNITS: I in A, but i in mA, V in mV, and B in mT

i = np.concatenate([iPos,iNeg])
err_i = Amprobe(i, unit="mA")
inter = np.concatenate([interPos,interNeg])
coeff = np.concatenate([coeffPos,coeffNeg]) # qui il coefficiente angolare: dovrà essere proporzionale a (i_p/t) * R_H

corrInter = []

# here we remove the ohmic contribution to check the compatibility of q with 0
ohm, err_ohm = ohmicContribution(i, err_i)

for j in range(0,len(inter)):
    a = inter[j][0] - ohm[j]
    b = np.sqrt(inter[j][1]**2 + err_ohm[j]**2)
    corrInter.append([a,b])

corrInter = np.array(corrInter)
    
z = Zscore(corrInter, np.array([np.zeros(2)]*len(corrInter)), Print =0)

for j in range(len(inter)):
    print(f" i = {i[j]} mA    intercetta: {corrInter[j]}  test z: {z[j]}")

# thus: R_H = 

RHplotter = fitPlotter("RH")
param1 = RHplotter.addGraph(i, coeff[:,0], err_i, coeff[:,1], title="Slopes against i; i [mA]; m [V/T]")
RHplotter.drawCanvas(legend=False)
RHplotter.saveCanvas("outputs/R_hfromB.png")

#print(param1[0][0], param1[0][1])
#print(param1[1][0], param1[1][1])

t     = 1   # mm
err_t = 0.1 # mm
R_H2 = param1[1][0] * t # mV * mm / (mT * mA) = m^3 / C
errR_H2 = np.sqrt((param1[1][1] * t)**2 + (param1[1][0] * err_t)**2)

rho2 = 1 / R_H2
err_rho2 = (errR_H2 / R_H2) * rho2

print(f"R_H = {R_H2} +- {errR_H2} [m^3 / C]")
print(f"rho = {rho2} +- {err_rho2} [C / m^3]")

 i = 2.008 mA    intercetta: [-0.00914274  0.01253396]  test z: 0.7294376385380638
 i = 4.0 mA    intercetta: [0.05505407 0.02106908]  test z: -2.6130267930615543
 i = 6.0 mA    intercetta: [-0.0500419  0.0315301]  test z: 1.5871153708751948
 i = -2.002 mA    intercetta: [-0.01743559  0.01052963]  test z: 1.6558606271866163
 i = -4.003 mA    intercetta: [0.03674307 0.0206548 ]  test z: -1.7789117842276982
 i = -6.005 mA    intercetta: [-0.04629289  0.03784179]  test z: 1.2233270384832906

--- fit Results for: Slopes against i; i [mA]; m [V/T] ---
Function: pol1
Chi2/NDF: 0.8096 / 4
p-value:  0.9372

p0: -0.0000 +/- 0.0001
p1: 0.0081 +/- 0.0000
--------------------------------
R_H = 0.008103422177060763 +- 0.0008113918747348903 [m^3 / C]
rho = 123.40465276890158 +- 12.356450197623268 [C / m^3]


Warning in <TCanvas::Constructor>: Deleting canvas with same name: RH
Info in <TCanvas::Print>: png file outputs/R_hfromB.png has been created


In [129]:
data = [inter[:,0], corrInter[:,0], coeff[:,0], i]
err  = [inter[:,1], corrInter[:,1], coeff[:,1], err_i]
lab  = ["q", "q_\\text{corr}", "m", "i"]

texTabler(data, err, lab)

$q$ & $\delta q$ & $q_\text{corr}$ & $\delta q_\text{corr}$ & $m$ & $\delta m$ & $i$ & $\delta i$ \\
\hline
0.103 & 0.012 & -0.009 & 0.013 & 0.0161 & 0.0002 & 2.008 & 0.015\\
0.28 & 0.02 & 0.06 & 0.02 & 0.0326 & 0.0003 & 4.00 & 0.03\\
0.28 & 0.03 & -0.05 & 0.03 & 0.0487 & 0.0005 & 6.00 & 0.03\\
-0.123 & 0.010 & -0.017 & 0.011 & -0.01628 & 0.00017 & -2.002 & 0.015\\
-0.18 & 0.02 & 0.04 & 0.02 & -0.0325 & 0.0003 & -4.00 & 0.03\\
-0.37 & 0.04 & -0.05 & 0.04 & -0.0484 & 0.0006 & -6.01 & 0.04\\


In [130]:
Zscore(np.array([[R_H1, errR_H1], [rho1, err_rho1]]), np.array([[R_H2, errR_H2], [rho2, err_rho2]]))

bestRH = MeanError(np.array([[R_H1, errR_H1]]), np.array([[R_H2, errR_H2]]))[0][0]
err_bestRH = MeanError(np.array([[R_H1, errR_H1]]), np.array([[R_H2, errR_H2]]))[0][1]
MeanError(np.array([[rho1, err_rho1]]), np.array([[rho2, err_rho2]]))

z value of param 0 : 0.429
z value of param 1 : -0.429




array([[126.99639379,   9.08351808]])

## Mobilità dei portatori

Misurando la caratteristica $I(V)$ del materiale seminconduttore (facendo variare la corrente $I$ tra -8mA e +8mA) che abbiamo usato nell'esperienza possiamo dare una stima della mobilità dei portatori di carica al suo interno. In particolare, dalla pendenza di $I(V)$ abbiamo la resistenza $R$, e quindi anche la resistività:
$$\rho =\frac{t\cdot w}{L} R$$
dove $t$ è lo spessore, $w$ la larghezza e $L$ la lunghezza.
Infine, dalla resistività si ha direttamente:
$$\mu = \frac{R_H}{\rho} (=R_H \sigma)$$
dove per $R_H$ possiamo considerare le stime date prima.

In [131]:
# fitting the curve I(V) as we need to find R in order tu calculate mu
# We miss 0 point

I   = np.array([-0.501, -1.01, -1.503, -1.999, -2.492, -3, -3.499, -4.004, -4.494, -5.002, -5.504, -6.018, -6.5, -6.998, -7.505, -8.002,0.537, 1.018, 1.514, 1.987, 2.499, 2.996, 3.498, 4.01, 4.49, 5.016, 5.451, 5.973, 6.47, 7, 7.5, 8]) #mA
errI = Amprobe(I, unit = "mA")
V    = [-32.505, -64.937, -97.343, -129.5, -161.399, -194.292, -226.477, -259.082, -290.841, -323.673, -356.178, -389.277, -420.475, -452.676, -484.506, -517.489,34.804, 65.991, 98.036, 128.64, 161.732, 193.9, 226.29, 259.353, 290.42, 324.444, 352.526, 386.29, 418.42, 452.78, 485.152, 517.532] #mV
errV = Keithley(V)

muPlotter = fitPlotter("IVcharacteristic")
paramR = muPlotter.addGraph(I, V, errI, errV, title="I(V); i [mA]; V [mV]")
muPlotter.drawCanvas(legend=False)
muPlotter.saveCanvas("outputs/IVcharacteristic.png")

R, errR = param[1][0],param[1][1]

# Z score for Delta
Zscore(np.array([[param[0][0], param[0][1]]]), np.array([[0,0]]))


--- fit Results for: I(V); i [mA]; V [mV] ---
Function: pol1
Chi2/NDF: 0.7654 / 30
p-value:  1.0000

p0: 0.0277 +/- 0.1969
p1: 64.6946 +/- 0.0746
--------------------------------
z value of param 0 : 0.00359




array([0.00359418])

Warning in <TCanvas::Constructor>: Deleting canvas with same name: IVcharacteristic
Info in <TCanvas::Print>: png file outputs/IVcharacteristic.png has been created


In [132]:
data = [I[0:16], V[0:16], I[16:33], V[16:33]]
errs = [errI[0:16], errV[0:16], errI[16:33], errV[16:33]]
labs = ["i_p", "V", "i_p", "V"]

texTabler(data, errs, labs)

$i_p$ & $\delta i_p$ & $V$ & $\delta V$ & $i_p$ & $\delta i_p$ & $V$ & $\delta V$ \\
\hline
-0.501 & 0.008 & -32.505 & 0.005 & 0.537 & 0.008 & 34.804 & 0.005\\
-1.010 & 0.010 & -64.937 & 0.006 & 1.018 & 0.010 & 65.991 & 0.006\\
-1.503 & 0.013 & -97.343 & 0.007 & 1.514 & 0.013 & 98.036 & 0.007\\
-1.999 & 0.015 & -129.500 & 0.010 & 1.987 & 0.015 & 128.640 & 0.010\\
-2.492 & 0.017 & -161.399 & 0.011 & 2.499 & 0.017 & 161.732 & 0.011\\
-3.00 & 0.02 & -194.292 & 0.012 & 3.00 & 0.02 & 193.900 & 0.012\\
-3.50 & 0.02 & -226.477 & 0.013 & 3.50 & 0.02 & 226.290 & 0.013\\
-4.00 & 0.03 & -259.082 & 0.014 & 4.01 & 0.03 & 259.353 & 0.014\\
-4.49 & 0.03 & -290.841 & 0.015 & 4.49 & 0.03 & 290.420 & 0.015\\
-5.00 & 0.03 & -323.673 & 0.016 & 5.02 & 0.03 & 324.444 & 0.016\\
-5.50 & 0.03 & -356.178 & 0.017 & 5.45 & 0.03 & 352.526 & 0.017\\
-6.02 & 0.04 & -389.277 & 0.018 & 5.97 & 0.03 & 386.290 & 0.018\\
-6.50 & 0.04 & -420.475 & 0.019 & 6.47 & 0.04 & 418.420 & 0.019\\
-7.00 & 0.04 & -452.68 & 0.02 & 7.00

/var/folders/1c/z7ldvl9s3q52971d3_jc4zf40000gn/T/ipykernel_3644/801023698.py:20: UserWarning: 2 significant figures requested from number with only 1 significant figures
  roundedString = round(data[j][i], errors[j][i], separation=' & ', cutoff=19)


In [133]:
# all measurements are in mm
import numpy as np

t = 1
err_t = 0.1
w = 10
err_w = 0.1
L = 20
err_L = 0.1

R = paramR[1][0]
err_R = paramR[1][1]

rho = t * w * R / L # ohm*mm
err_rho = np.sqrt((((w*R)/L)*err_t)**2 + (((t*R)/L)*err_w)**2 + (((t*w)/L)*err_L)**2 + (((-t*w*R)/L**2)*err_R)**2) 

print(rho, err_rho)

mu = bestRH / rho
err_mu = np.sqrt(((1/rho)*err_bestRH)**2 + ((-bestRH/rho**2)*err_rho)**2)

print(mu, err_mu)

32.34732495331314 3.2534909803044254
0.00024274123201011082 2.9958899181731098e-05


### facoltativo: Magnetoresistenza

In [134]:
# we want to study the curve I(V) for different B values
# i is measured in mA while V in mV

i1 = np.array([0, 2.002, 4.018, 6.01, 8.013]) 
V1 = np.array([0.007, 128.586, 257.906, 385.834, 514.379])
erri1 = Amprobe(i1, unit = "mA")
errV1 = Keithley(V1)

i2 = np.array([0, 2.002, 4.001, 6.004, 8])
V2 = np.array([0.016, 128.974, 257.59, 386.506, 515.038])
erri2 = Amprobe(i2, unit = "mA")
errV2 = Keithley(V2)

i3 = np.array([0, 2.007, 4.006, 6.007, 8.004])
V3 = np.array([0.009, 130.183, 259.769, 389.619, 519.252])
erri3 = Amprobe(i3, unit = "mA")
errV3 = Keithley(V3)

i4 = np.array([0, 2.007, 4.006, 6.024, 8.008])
V4 = np.array([0.01, 131.567, 262.519, 394.712, 524.736])
erri4 = Amprobe(i4, unit = "mA")
errV4 = Keithley(V4)

i5 = np.array([0, 2.005, 4.002, 6.006, 8.017])
V5 = np.array([0.007, 132.956, 265.213, 397.935, 531.45])
erri5 = Amprobe(i5, unit = "mA")
errV5 = Keithley(V5)

i6 = np.array([0, 2.002, 4.02, 5.999, 7.992])
V6 = np.array([0.007, 134.11, 269.21, 401.676, 535.128])
erri6 = Amprobe(i6, unit = "mA")
errV6 = Keithley(V6) 

IVplotter = fitPlotter("Current Vs Voltage")

param1pos = IVplotter.addGraph(i1,V1,erri1,errV1, title="Magnet Current 0.000 A; i [mA]; V [mV]")
param2pos = IVplotter.addGraph(i2,V2,erri2,errV2, title="Magnet Current 0.400 A; i [mA]; V [mV]")
param3pos = IVplotter.addGraph(i3,V3,erri3,errV3, title="Magnet Current 0.795 A; i [mA]; V [mV]")
param4pos = IVplotter.addGraph(i4,V4,erri4,errV4, title="Magnet Current 1.200 A; i [mA]; V [mV]")
param5pos = IVplotter.addGraph(i5,V5,erri5,errV5, title="Magnet Current 1.595 A; i [mA]; V [mV]")
param6pos = IVplotter.addGraph(i6,V6,erri6,errV6, title="Magnet Current 2.001 A; i [mA]; V [mV]")

paramRes  = np.array([param1pos, param2pos, param3pos, param4pos, param5pos, param6pos])

IVplotter.drawCanvas(legend=False)
IVplotter.saveCanvas("outputs/I(V).png")


--- fit Results for: Magnet Current 0.000 A; i [mA]; V [mV] ---
Function: pol1
Chi2/NDF: 0.0044 / 3
p-value:  0.9999

p0: 0.0070 +/- 0.0036
p1: 64.1978 +/- 0.1986
--------------------------------

--- fit Results for: Magnet Current 0.400 A; i [mA]; V [mV] ---
Function: pol1
Chi2/NDF: 0.0055 / 3
p-value:  0.9999

p0: 0.0160 +/- 0.0036
p1: 64.3823 +/- 0.1993
--------------------------------

--- fit Results for: Magnet Current 0.795 A; i [mA]; V [mV] ---
Function: pol1
Chi2/NDF: 0.0031 / 3
p-value:  1.0000

p0: 0.0090 +/- 0.0036
p1: 64.8595 +/- 0.2007
--------------------------------

--- fit Results for: Magnet Current 1.200 A; i [mA]; V [mV] ---
Function: pol1
Chi2/NDF: 0.0022 / 3
p-value:  1.0000

p0: 0.0100 +/- 0.0036
p1: 65.5292 +/- 0.2028
--------------------------------

--- fit Results for: Magnet Current 1.595 A; i [mA]; V [mV] ---
Function: pol1
Chi2/NDF: 0.0088 / 3
p-value:  0.9998

p0: 0.0070 +/- 0.0036
p1: 66.2779 +/- 0.2051
--------------------------------

--- fit Result

Warning in <TCanvas::Constructor>: Deleting canvas with same name: Current Vs Voltage
Info in <TCanvas::Print>: png file outputs/I(V).png has been created


In [135]:
data = [V1, i1, V2, i2, V3, i3, V4, i4, V5, i5, V6, i6]
errs = [errV1, erri1, errV2, erri2, errV3, erri3, errV4, erri4, errV5, erri5, errV6, erri6]
labs = ["V_1", "i_1", "V_2", "i_2", "V_3", "i_3", "V_4", "i_4", "V_5", "i_5", "V_6", "i_6"]

texTabler(data[0:6], errs[0:6], labs[0:6])
texTabler(data[6:12], errs[6:12], labs[6:12])

$V_1$ & $\delta V_1$ & $i_1$ & $\delta i_1$ & $V_2$ & $\delta V_2$ & $i_2$ & $\delta i_2$ & $V_3$ & $\delta V_3$ & $i_3$ & $\delta i_3$ \\
\hline
0.007 & 0.004 & 0.000000 & 0.000010 & 0.016 & 0.004 & 0.000000 & 0.000010 & 0.009 & 0.004 & 0.000000 & 0.000010\\
128.586 & 0.010 & 2.002 & 0.015 & 128.974 & 0.010 & 2.002 & 0.015 & 130.183 & 0.010 & 2.007 & 0.015\\
257.906 & 0.014 & 4.02 & 0.03 & 257.590 & 0.014 & 4.00 & 0.03 & 259.769 & 0.014 & 4.01 & 0.03\\
385.834 & 0.018 & 6.01 & 0.04 & 386.506 & 0.018 & 6.00 & 0.04 & 389.619 & 0.018 & 6.01 & 0.04\\
514.38 & 0.02 & 8.01 & 0.05 & 515.04 & 0.02 & 8.00 & 0.05 & 519.25 & 0.02 & 8.00 & 0.05\\
$V_4$ & $\delta V_4$ & $i_4$ & $\delta i_4$ & $V_5$ & $\delta V_5$ & $i_5$ & $\delta i_5$ & $V_6$ & $\delta V_6$ & $i_6$ & $\delta i_6$ \\
\hline
0.010 & 0.004 & 0.000000 & 0.000010 & 0.007 & 0.004 & 0.000000 & 0.000010 & 0.007 & 0.004 & 0.000000 & 0.000010\\
131.567 & 0.010 & 2.007 & 0.015 & 132.956 & 0.010 & 2.005 & 0.015 & 134.110 & 0.010 & 2.002 & 0.

Per fittare i dati sperimentali ho scelto di usare la relazione $$ R(B) = R_0 + ( \mu B )^2 $$ dove $R_0$ è il valore della resistenza a campo magnetico spento mentre $\mu$ è la mobilità dei portatori

In [136]:
# now we study the R(B) curve using the parameters found in the cell above
# I'm using the R values found in the fits above

R = paramRes[:,1,0] #ohm
I_mag = np.array([0,0.4,0.795,1.2,1.595,2.001]) # A
err_R = paramRes[:,1,1]
errI_mag = Amprobe(I_mag, unit = "A")

B, err_B = hysteresisMean(I_mag,errI_mag)

RBplotter = fitPlotter("Resistance VS B")

# we expect mu_new = mu_p / (1+mu_p^2): it seems to make sense
parampos = RBplotter.addGraph(B/1000,R,err_B/1000,err_R, title = "Magnetoresistance; B [T]; R [#Omega]", fit_formula = "[0]+[1]*x^2",setparam = np.array([64,12]))
                                                                                                              
RBplotter.drawCanvas(legend=False)
RBplotter.saveCanvas("outputs/magnetoresistance.png")


--- fit Results for: Magnetoresistance; B [T]; R [#Omega] ---
Function: [0]+[1]*x^2
Chi2/NDF: 2.3268 / 4
p-value:  0.6759

p0: 64.3263 +/- 0.1223
p1: 12.5757 +/- 1.2369
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: Resistance VS B
Info in <TCanvas::Print>: png file outputs/magnetoresistance.png has been created


In [137]:
data = [R, paramRes[:,0,0], I_mag, B]
errs = [err_R, paramRes[:,0,1], errI_mag, err_B]
labs = ["R", "\\Delta", "I", "B"]

texTabler(data, errs, labs)

$R$ & $\delta R$ & $\Delta$ & $\delta \Delta$ & $I$ & $\delta I$ & $B$ & $\delta B$ \\
\hline
64.2 & 0.2 & 0.007 & 0.004 & 0.000000000 & 0.000000010 & 0 & 9\\
64.4 & 0.2 & 0.016 & 0.004 & 0.400 & 0.016 & 95 & 10\\
64.9 & 0.2 & 0.009 & 0.004 & 0.80 & 0.02 & 190 & 10\\
65.5 & 0.2 & 0.010 & 0.004 & 1.20 & 0.03 & 286 & 11\\
66.3 & 0.2 & 0.007 & 0.004 & 1.60 & 0.03 & 380 & 12\\
67.0 & 0.2 & 0.007 & 0.004 & 2.00 & 0.04 & 477 & 14\\
